# Cluster-based recommendation system

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import yaml

standardscaler = StandardScaler()



In [2]:
with open('/home/uzokmurod/Desktop/amaliyot/book recommendation sysrem/config/config.yaml', 'r') as file:
    config = yaml.safe_load(file)

ratings = pd.read_csv(config['data']['url_rating'])
books = pd.read_csv(config['data']['url_books'])
tags = pd.read_csv(config['data']['url_tags'])

In [3]:
ratings

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4
...,...,...,...
981751,10000,48386,5
981752,10000,49007,4
981753,10000,49383,5
981754,10000,50124,5


In [4]:
ratings.groupby('user_id').size()

user_id
1         3
2         3
3         2
4         3
5         5
         ..
53420     6
53421     8
53422    18
53423     2
53424    16
Length: 53424, dtype: int64

In [5]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 981756 entries, 0 to 981755
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   book_id  981756 non-null  int64
 1   user_id  981756 non-null  int64
 2   rating   981756 non-null  int64
dtypes: int64(3)
memory usage: 22.5 MB


In [6]:
ratings.isna().sum()

book_id    0
user_id    0
rating     0
dtype: int64

In [7]:
user_book_matrix = ratings.pivot_table(index='user_id', columns='book_id', values='rating').fillna(0)

In [ ]:


# 4️⃣ Normalize the matrix (important for clustering)
scaler = StandardScaler()
user_scaled = scaler.fit_transform(user_book_matrix)

# 5️⃣ Clustering users using KMeans
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
user_clusters = kmeans.fit_predict(user_scaled)

# 6️⃣ Add cluster labels to the users
user_cluster_df = pd.DataFrame({
    'user_id': user_book_matrix.index,
    'cluster': user_clusters
})

# 7️⃣ Merge the cluster info with original ratings
ratings_with_cluster = ratings.merge(user_cluster_df, on='user_id')

# 8️⃣ Count how many users in each cluster rated each book
cluster_popularity = ratings_with_cluster.groupby(['cluster', 'book_id']).agg(
    popularity=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

# 9️⃣ Define a function to recommend books for a specific user
def recommend_books(user_id, top_n=10):
    if user_id not in user_cluster_df['user_id'].values:
        return f"User ID {user_id} not found in dataset."

    # Get the user's cluster
    cluster_id = user_cluster_df[user_cluster_df['user_id'] == user_id]['cluster'].iloc[0]

    # Books the user has already rated
    user_read_books = ratings[ratings['user_id'] == user_id]['book_id'].tolist()

    # Books popular in the user's cluster
    cluster_books = cluster_popularity[cluster_popularity['cluster'] == cluster_id]

    # Filter out books already read
    recommendations = cluster_books[cluster_books['book_id'].isin(user_read_books)]

    # Sort by popularity and average rating
    recommendations = recommendations.sort_values(['popularity', 'avg_rating'], ascending=False)

    # Get book titles
    recommendations = recommendations.merge(books[['book_id', 'title']], on='book_id')

    return recommendations[['book_id', 'title', 'popularity', 'avg_rating']].head(top_n)

user_id = 31
recommended_books = recommend_books(user_id, top_n=5)
print("📚 Recommended Books for User", user_id)
print(recommended_books)